In [ ]:
import glob
import ast
import time
import numpy as np
import pandas as pd
from tqdm import tqdm 
import cv2
from PIL import Image
import re
import joblib
import matplotlib.pyplot as plt
import gc
import zipfile

from tqdm import tqdm
import shutil

In [ ]:
import warnings
warnings.filterwarnings('ignore')

### Loading package

In [ ]:
import sys
from pathlib import Path

here_path = Path().resolve()
repo_path = here_path.parents[2]
sys.path.append(str(repo_path))

In [ ]:
from py.utils import verifyDir,verifyFile

In [ ]:
from py.config import Config

cfg = Config()

np.random.seed(cfg.RANDOM_STATE)
cfg.DATA_PATH, cfg.MODEL_PATH

In [ ]:
QSCORE_PATH=f"{cfg.DATA_PATH}pp2/Qscores/"
IMAGES_PATH = f"{cfg.DATA_PATH}pp2/images/"

SEG_DATASET_DIR = f"{cfg.DATA_PATH}{cfg.SEG_DATASET}/"
UPD4K_DIR = f"{cfg.DATA_PATH}/{cfg.UPD_DATASET}/"

UPD4k_PATH = f"{cfg.DATA_PATH}UrbanPhysicalDisorder/{cfg.UPD_DATASET}/"
GROUP_UPD4k_PATH = f"{cfg.DATA_PATH}UrbanPhysicalDisorder/{cfg.UPD_DATASET}_group/"

In [ ]:
verifyDir(GROUP_UPD4k_PATH)

### Loading data

In [ ]:
%%time
data_df = pd.read_csv(f"{QSCORE_PATH}scores.csv", sep=";", low_memory=False)
cities = list(np.sort(data_df["city"].unique()))

### Grouping Segmentations and  Disorders

In [ ]:
from py.datasets.segmentation import UPD4KDataset

uss = UPD4KDataset(data_path=UPD4K_DIR)
uss.load(dataset=cfg.SEG_DATASET, data_path=SEG_DATASET_DIR)

In [ ]:
group_df = uss.get_urban_street_categories(by_groups=True)
group_df

In [ ]:
vectorized_map = uss.get_vectorized_map(group_df)

In [ ]:
from py.datasets.segmentation import MaskProcessor, FeatureProcessor

mask_processor = MaskProcessor()
feature_processor = FeatureProcessor()

### Group Segmentations + UPD

In [ ]:
%%time

upd_merge_segment_df = pd.DataFrame()

for idx, current_city in enumerate(["Rio De Janeiro"]):
    upd4k_masks = np.sort(glob.glob(f'{UPD4k_PATH}/{current_city}/masks/*.pkl'))
    if len(upd4k_masks)==0:
        continue
    print(f"{idx+1}: Grouping {current_city}...")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/masks/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/ratios/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images/")
    verifyDir(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images_overlay/")

    if verifyFile(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv"):
        seg_upd4k_df = pd.read_csv(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", low_memory=False)
        upd_merge_segment_df = pd.concat([upd_merge_segment_df, seg_upd4k_df], ignore_index=True)
        upd_merge_segment_df.fillna(0, inplace=True)
        continue
    
    seg_upd4k_df = pd.read_csv(f"{UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", low_memory=False)
    seg_upd4k_df = feature_processor.process(seg_upd4k_df, 
                                             class_to_group=uss.get_class_groups(), 
                                             aggregate_classes=True, 
                                             filter_features=True)
    seg_upd4k_df.drop(columns=["indoor_object", "outdoor_object", "nature_object"], inplace=True)
    seg_upd4k_df.to_csv(f"{GROUP_UPD4k_PATH}/{current_city}/segmentations.csv", sep=";", index=False)

    upd_merge_segment_df = pd.concat([upd_merge_segment_df, seg_upd4k_df], ignore_index=True)
    upd_merge_segment_df.fillna(0, inplace=True)
    
    for cur_mask in tqdm(upd4k_masks):
        current_id = cur_mask.split("/")[-1]
        image_name = current_id.replace(".pkl", "")
        current_image = Image.open(f'{IMAGES_PATH}/{current_city}/{image_name}.JPG'  ).convert("RGB")

        # UPD mask
        usd_mask = joblib.load(cur_mask)
        usd_new_mask = vectorized_map(usd_mask)
        joblib.dump(usd_new_mask, f"{GROUP_UPD4k_PATH}/{current_city}/masks/{current_id}")

        # Group ratio
        unique, counts = np.unique(usd_new_mask, return_counts=True)
        total = usd_new_mask.size
        proportions = {int(k): float(v / total) * 100 for k, v in zip(unique, counts)}
        df = pd.DataFrame(list(proportions.items()), columns=["group_class_id", "ratio"])
        usd_ratio_df = pd.merge(df, group_df[["group_class_name", "RGB_color_group", "hex_color_group", "group_class_id"]], on="group_class_id", how="left")
        usd_ratio_df.to_csv(f"{GROUP_UPD4k_PATH}/{current_city}/ratios/{image_name}.csv", sep=";", index=False)

        # mask
        usd_new_image = mask_processor.convert_matrix_to_mask(usd_new_mask, uss.get_color_dict(by_group=True))
        usd_new_image.save(f"{GROUP_UPD4k_PATH}{current_city}/segmented_images/{image_name}.png")

        # overlay
        orig_usd_overlay = Image.blend(current_image, usd_new_image, alpha=0.6)
        orig_usd_overlay.save(f"{GROUP_UPD4k_PATH}/{current_city}/segmented_images_overlay/{image_name}.png")


In [ ]:
upd_merge_segment_df

In [ ]:
upd_merge_segment_df.to_csv(f"{GROUP_UPD4k_PATH}/segmentations.csv", sep=";", index=False)